# Selective State Space Layer (Mamba-style, simplified)

Companion notebook for the [Beyond Transformers](https://codemia.io/courses/llm_inside_and_out/beyond_transformers) lesson on Codemia.

**What you will build**: a simplified selective state-space layer where the B and C matrices are produced from the input (the "selection" mechanism in Mamba), train a tiny char-LM on Shakespeare, and compare training throughput and validation loss against an equivalently-sized transformer.

**Note**: this is an *educational* implementation. It runs the recurrence in a Python loop over time, which is O(T) but slow in wall clock. A production Mamba uses a hardware-aware parallel scan (the "S6" kernel). The loop version is what you want for understanding the math.

**Time**: ~30-60 minutes (training is short on T4)

**GPU required**: yes (T4 is enough)

## Setup

In [ ]:
%pip install -q torch numpy matplotlib requests

In [ ]:
import math, time, requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Concept recap

A linear state-space model (SSM) treats a sequence as a continuous-time signal filtered through a linear system:

```
h_t = A h_{t-1} + B x_t        # state update
y_t = C h_t + D x_t            # output
```

This is just a linear RNN. What makes Mamba "selective" is that `B` and `C` (and the discretization step `delta`) are *functions of the input* `x_t`. Before Mamba, S4 used fixed `A, B, C` so the recurrence could be computed in parallel as a convolution. Selectivity breaks that trick but gives the model the ability to "pay attention" in a different way: it decides, per token, how much to read from the input and how much of the state to emit.

We implement the time loop directly. Every step reads the current input, computes input-dependent `B_t` and `C_t`, updates the state, and emits an output.

## Data: tiny Shakespeare

In [ ]:
# Karpathy's canonical tiny-shakespeare URL. 1.1 MB. Downloaded once and cached.
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = requests.get(url).text
print(f"{len(text):,} chars, preview:")
print(text[:200])

In [ ]:
# Build a character vocab. Small enough (~65 chars) that one-hot lookup is free.
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
print('vocab_size:', vocab_size)

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train: {len(train_data):,}  val: {len(val_data):,}")

In [ ]:
def get_batch(split, batch_size, block_size):
    d = train_data if split == 'train' else val_data
    # Random starting offsets into the stream. Classic char-LM batching.
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

## Implementation: selective SSM layer

In [ ]:
class SelectiveSSM(nn.Module):
    # One selective-SSM layer. State size d_state, hidden dim d_model.
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # A is kept *input-independent* for stability (as in Mamba).
        # We parameterize log(-A) so A is always negative, making the recurrence stable.
        # Each hidden channel has its own d_state-dimensional A vector -- a diagonal A matrix per channel.
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).repeat(d_model, 1))

        # B, C, and delta come from input-conditioned linear maps. This is the "selection".
        self.x_proj = nn.Linear(d_model, d_state * 2 + 1, bias=False)
        # delta gets its own learned scale and bias so it has room to grow.
        self.dt_proj = nn.Linear(1, d_model, bias=True)

        # D is a learned skip connection (the "x_t feed-through" in the SSM equation).
        self.D = nn.Parameter(torch.ones(d_model))

        # Input / output projections (the "gate" + "in/out" linears in the Mamba paper).
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x: (B, T, d_model)
        B, T, _ = x.shape
        # Gating branch (silu) and main branch, same as Mamba.
        xz = self.in_proj(x)                           # (B, T, 2*d_model)
        x_main, z = xz.chunk(2, dim=-1)                # each (B, T, d_model)
        x_main = F.silu(x_main)

        # Per-token selection: compute delta, B, C from the input.
        proj = self.x_proj(x_main)                     # (B, T, 2*d_state + 1)
        dt = proj[..., :1]                             # (B, T, 1)
        B_t = proj[..., 1:1+self.d_state]              # (B, T, d_state)
        C_t = proj[..., 1+self.d_state:]               # (B, T, d_state)
        # Expand delta to d_model channels and pass through softplus so it's positive.
        dt = F.softplus(self.dt_proj(dt))              # (B, T, d_model)

        # A is (d_model, d_state), positive: -exp(A_log).
        A = -torch.exp(self.A_log)                     # (d_model, d_state)

        # Discretize: dA = exp(dt * A). Per batch, per time, per channel, per state.
        # Shape gymnastics: dt is (B, T, d_model), A is (d_model, d_state).
        dA = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))   # (B, T, d_model, d_state)
        # dB = dt * B. Approximation of the zero-order hold discretization.
        dB = dt.unsqueeze(-1) * B_t.unsqueeze(-2)                         # (B, T, d_model, d_state)

        # Recurrence: loop over time. Slow in Python -- this is the educational version.
        h = torch.zeros(B, self.d_model, self.d_state, device=x.device)
        outputs = []
        for t in range(T):
            # State update: h <- dA * h + dB * x_main.
            h = dA[:, t] * h + dB[:, t] * x_main[:, t].unsqueeze(-1)
            # Output: y = C . h + D * x_main. C is per-token (selection).
            y_t = (h * C_t[:, t].unsqueeze(1)).sum(dim=-1) + self.D * x_main[:, t]
            outputs.append(y_t)
        y = torch.stack(outputs, dim=1)   # (B, T, d_model)

        # Gate and project out, exactly like Mamba's block.
        return self.out_proj(y * F.silu(z))

## Implementation: wrap it in a language model

In [ ]:
class SSMLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2, d_state=16):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleList([nn.LayerNorm(d_model), SelectiveSSM(d_model, d_state=d_state)])
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        # Residual stream through each SSM layer; LN before the layer as in pre-LN transformers.
        for norm, ssm in self.layers:
            x = x + ssm(norm(x))
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## Implementation: a matching transformer baseline

In [ ]:
class TinyTransformerLM(nn.Module):
    # Same vocab, same embedding dim, same layer count. Different mixer.
    def __init__(self, vocab_size, d_model=64, n_layers=2, n_head=4, block_size=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_head, dim_feedforward=4*d_model,
            dropout=0.0, activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None]
        # Causal mask for self-attention (same convention as transformer_block.ipynb).
        mask = torch.triu(torch.full((T, T), float('-inf'), device=idx.device), diagonal=1)
        x = self.encoder(x, mask=mask, is_causal=True)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## Experiment: train both for a fixed wall-clock budget

Same vocab, same embedding dim, same layer count. We train each model for 400 steps and record loss + steps/sec. Keeping the block size short (64) so the Python-loop SSM is not catastrophically slow.

In [ ]:
block_size = 64
batch_size = 32
d_model = 64
n_layers = 2
n_steps = 400
eval_every = 50

def train_model(model, label):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    losses = []
    t0 = time.time()
    for step in range(n_steps):
        xb, yb = get_batch('train', batch_size, block_size)
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % eval_every == 0:
            # Quick val loss snapshot so we can plot curves on matched x-axes.
            model.eval()
            with torch.no_grad():
                xv, yv = get_batch('val', batch_size, block_size)
                _, vloss = model(xv, yv)
            model.train()
            print(f"{label:<11} step {step:4d}  train {loss.item():.3f}  val {vloss.item():.3f}")
    dt = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{label}: {n_steps} steps in {dt:.1f}s, {n_steps/dt:.1f} steps/s, {n_params:,} params")
    return losses, dt, n_params

In [ ]:
print("---- Selective SSM ----")
ssm_model = SSMLanguageModel(vocab_size, d_model=d_model, n_layers=n_layers)
ssm_losses, ssm_time, ssm_params = train_model(ssm_model, 'ssm')

print()
print("---- Transformer ----")
tfm_model = TinyTransformerLM(vocab_size, d_model=d_model, n_layers=n_layers, block_size=block_size)
tfm_losses, tfm_time, tfm_params = train_model(tfm_model, 'transformer')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ssm_losses, label='selective SSM', alpha=0.8)
ax.plot(tfm_losses, label='transformer', alpha=0.8)
ax.set_xlabel('step')
ax.set_ylabel('train loss')
ax.set_title('Char-level Shakespeare, d_model=64, 2 layers')
ax.legend()
plt.tight_layout()
plt.show()

print(f"transformer: {tfm_time:5.1f}s  ({tfm_params:,} params)")
print(f"ssm (loop):  {ssm_time:5.1f}s  ({ssm_params:,} params)")
print()
print("The transformer is faster in wall clock at this scale -- its forward pass")
print("is all batched matmuls on GPU, while our educational SSM runs a Python loop.")
print("Real Mamba uses a parallel scan kernel that closes the gap.")

## Variation: compare validation perplexity at the end

In [ ]:
@torch.no_grad()
def estimate_val_loss(model, n_batches=10):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xb, yb = get_batch('val', batch_size, block_size)
        _, l = model(xb, yb)
        losses.append(l.item())
    model.train()
    return sum(losses) / len(losses)

ssm_val = estimate_val_loss(ssm_model)
tfm_val = estimate_val_loss(tfm_model)
print(f"ssm         val loss {ssm_val:.3f}  perplexity {math.exp(ssm_val):.2f}")
print(f"transformer val loss {tfm_val:.3f}  perplexity {math.exp(tfm_val):.2f}")

## Sample from each model

In [ ]:
@torch.no_grad()
def sample(model, start="ROMEO:", max_new=200):
    model.eval()
    idx = torch.tensor([[stoi[c] for c in start]], device=device)
    for _ in range(max_new):
        # Clip to block size for models with fixed positions (the transformer).
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1], dim=-1)
        nxt = torch.multinomial(probs, 1)
        idx = torch.cat([idx, nxt], dim=1)
    model.train()
    return ''.join(itos[i.item()] for i in idx[0])

print("-- selective SSM --")
print(sample(ssm_model))
print()
print("-- transformer --")
print(sample(tfm_model))

## What to try next

- Replace the Python time loop with `torch.associative_scan` (PyTorch 2.4+) or with a hand-rolled parallel scan. Wall clock should improve dramatically.
- Initialize `A` using the HiPPO formula from S4 and see if it changes the learned dynamics.
- Stack more layers (4 or 6) and train for 2000 steps. Does the SSM start to match the transformer on perplexity?
- Read the Mamba paper and identify the three "selection mechanisms" that make S6 different from S4.